# Foot detector â€” SynFoot 25K subset (Colab GPU)

Same pipeline as `colab_synfoot_train.ipynb` but trains on ~25K of the 43,742 train images
via `fraction=0.571` (25,000 / 43,742). Use this when you want a faster/cheaper pretrain
before committing to the full 100-epoch 50K run. Val still runs on the full 6,258 val split.
Expect ~15-30 min on a T4/A100 instead of ~30-60 min.

In [ ]:
# 0) Mount Drive â€” checkpoints get backed up here so a GPU-limit kill doesn't lose them
from google.colab import drive
drive.mount('/content/drive')
BACKUP_DIR = '/content/drive/MyDrive/foot-detector-train'
!mkdir -p "$BACKUP_DIR"
print(BACKUP_DIR)

In [ ]:
!nvidia-smi -L
!python -c "import torch; print('cuda:', torch.cuda.is_available())"
!pip install ultralytics onnx onnxruntime albumentations

In [ ]:
# Restore repo files from Drive if this is a fresh VM (Reconnective runtimes wipe /content).
# Upload once: data-synfoot.yaml + synfoot-50k.zip under MyDrive/foot-detector-train/
import os
os.makedirs('training/foot-detector', exist_ok=True)
!cp -n "/content/drive/MyDrive/foot-detector-train/data-synfoot.yaml" training/foot-detector/ 2>/dev/null || echo "yaml already in repo, skipping cp"
!if [ ! -d training/foot-detector/datasets/synfoot/images/train ]; then unzip -q "/content/drive/MyDrive/foot-detector-train/synfoot-50k.zip" -d training/foot-detector/; fi
!python -c "import os; b='training/foot-detector/datasets/synfoot'; print({s: {k: len(os.listdir(os.path.join(b,k,s))) for k in ['images','labels']} for s in ['train','val']})"
# expect train 43742 / val 6258; fraction=0.571 below uses ~25K of train

In [ ]:
# Ultralytics resolves a relative `path:` against its own datasets folder,
# not against this yaml â€” rewrite it to the absolute VM path or train fails
# with a doubled `datasets/datasets/...` lookup. Idempotent: safe to re-run.
p = 'training/foot-detector/data-synfoot.yaml'
t = open(p).read()
if 'path: datasets/synfoot' in t:
    t = t.replace('path: datasets/synfoot', 'path: /content/training/foot-detector/datasets/synfoot')
    open(p, 'w').write(t)
print(open(p).read())

In [ ]:
# Find any interrupted checkpoint on this VM (e.g. killed at epoch 38/100 by GPU limits)
!find /content -name "last.pt" -o -name "best.pt" 2>/dev/null | head -20

In [ ]:
# Pretrain on SynFoot-25k. If a last.pt exists from an interrupted run, resume it;
# otherwise start fresh. Resume keeps epoch counter (39/100, not 1/100).
import glob, os
cands = sorted(glob.glob('/content/**/synfoot-25k/weights/last.pt', recursive=True))
cands += sorted(glob.glob('training/foot-detector/runs/synfoot-25k/weights/last.pt'))
RESUME = cands[-1] if cands else ''
print('resume_from:', RESUME or '(fresh start)')
if RESUME:
    print('To resume, run the NEXT cell (resume=True). Skip the fresh-train cell below.')
else:
    print('No checkpoint found â€” run the fresh-train cell below.')

In [ ]:
# FRESH train â€” only run this when there is NO checkpoint to resume
!yolo pose train \
  data=training/foot-detector/data-synfoot.yaml \
  model=yolo11n-pose.pt \
  epochs=100 imgsz=640 batch=16 fraction=0.571 \
  hsv_h=0.015 hsv_s=0.4 hsv_v=0.4 \
  degrees=15 translate=0.1 scale=0.3 \
  mosaic=0.5 mixup=0.1 copy_paste=0.1 seed=0 \
  project=training/foot-detector/runs name=synfoot-25k

In [ ]:
# RESUME train â€” only run this when the cell above found a last.pt (e.g. after GPU-limit kill).
# Set RESUME_PT to the path printed two cells up, then run.
RESUME_PT = ''  # e.g. 'training/foot-detector/runs/synfoot-25k/weights/last.pt'
# !yolo pose train resume=True model="$RESUME_PT"
# Equivalent explicit form (same args as fresh, Ultralytics picks up the epoch):
# !yolo pose train data=training/foot-detector/data-synfoot.yaml model="$RESUME_PT" resume=True epochs=100 imgsz=640 batch=16 fraction=0.571 hsv_h=0.015 hsv_s=0.4 hsv_v=0.4 degrees=15 translate=0.1 scale=0.3 mosaic=0.5 mixup=0.1 copy_paste=0.1 seed=0 project=training/foot-detector/runs name=synfoot-25k

In [ ]:
# Back up checkpoints to Drive IMMEDIATELY after train â€” runtimes can be reclaimed anytime
!find /content -path "*synfoot-25k/weights/*.pt" 2>/dev/null | head
!cp training/foot-detector/runs/synfoot-25k/weights/last.pt "$BACKUP_DIR/synfoot-25k-last.pt" 2>/dev/null || cp /content/runs/pose/training/foot-detector/runs/synfoot-25k/weights/last.pt "$BACKUP_DIR/synfoot-25k-last.pt"
!cp training/foot-detector/runs/synfoot-25k/weights/best.pt "$BACKUP_DIR/synfoot-25k-best.pt" 2>/dev/null || cp /content/runs/pose/training/foot-detector/runs/synfoot-25k/weights/best.pt "$BACKUP_DIR/synfoot-25k-best.pt"
!ls -lh "$BACKUP_DIR/" | grep -E "last|best"

In [ ]:
!yolo pose val data=training/foot-detector/data-synfoot.yaml model=training/foot-detector/runs/synfoot-25k/weights/best.pt

In [ ]:
!yolo export model=training/foot-detector/runs/synfoot-25k/weights/best.pt format=onnx opset=12 dynamic=False
!ls -lh training/foot-detector/runs/synfoot-25k/weights/